# RAG 개요, LLM 한계와 "외부 지식" 보강
- LLM 은 학습 시점 이후의 정보를 모르고, 우리 회사 내부 문서 같은 비공개 정보도 모름
- 매번 시스템 프롬프트에 모든 정보를 넣는 건 토큰 한계 때문에 불가능
- **RAG (Retrieval-Augmented Generation)** = "질문에 맞는 정보를 외부에서 찾아 와서 LLM 에 같이 주입"

## RAG 시스템 구축 순서

1. **Load** 문서 가져오기 (PDF, MD, HTML, DB)
2. **Split** 청크로 자르기 (LLM 컨텍스트에 맞게)
3. **Embed** 각 청크를 벡터로 (의미 검색용)
4. **Store** 벡터 DB 에 저장
5. **Retrieve & Generate** 질문을 벡터로 → 유사한 청크 검색 → LLM 답변


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [12]:
# 필요한 라이브러리 설치
# uv add langchain-openai numpy scikit-learn

## (2) 라이브러리 Import

이번 실습에서 사용하는 핵심 객체는 다음과 같습니다.

In [13]:
from dotenv import load_dotenv
load_dotenv()

True

## 2. "비슷한 문장" 을 컴퓨터가 어떻게 알까
- LLM 은 텍스트를 **벡터(숫자 배열)** 로 바꿔서 비교함
- 의미가 비슷하면 벡터 거리가 가까움

In [14]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

sentences = [
    '마법사는 마나를 다루는 직업이다.',
    '위자드는 마법 에너지로 싸우는 캐릭터다.',
    '오늘 점심에 김치찌개를 먹어야지'
]

vectors = embeddings.embed_documents(sentences)
print(f'벡터 차원: {len(vectors[0])}')          # 

벡터 차원: 1536


## 3. 코사인 유사도, "두 벡터가 얼마나 같은 방향을 가리키나"

In [15]:
def cosine_sim(a, b):
    """코사인 유사도. 1 = 같은 방향(=의미가 같음), 0 = 무관, -1 = 정반대."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print(f"[마법사 vs 위자드]  {cosine_sim(vectors[0], vectors[1]):.3f}")
print(f"[마법사 vs 김치찌개]  {cosine_sim(vectors[0], vectors[2]):.3f}")
print(f"[위자드 vs 김치찌개]  {cosine_sim(vectors[1], vectors[2]):.3f}")

[마법사 vs 위자드]  0.314
[마법사 vs 김치찌개]  0.114
[위자드 vs 김치찌개]  0.091


## 4. 질문도 같은 방식으로 벡터화

In [16]:
docs = [
    "신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.",
    "법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.",
    "개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.",
    "장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.",
    "재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.",
]

doc_vectors = embeddings.embed_documents(docs)

In [17]:
question = '법인카드 영수증은 언제까지 제출해야 하나요?'
q_vector = embeddings.embed_query(question)

In [18]:
# 질문 vs 각 문서 유사도
scores = [cosine_sim(q_vector, dv) for dv in doc_vectors]
ranked = sorted(zip(scores, docs), reverse=True)

print(f"질문: {question}\n")
print("=== 유사도 순위 ===")
for score, doc in ranked:
    print(f"  {score:.3f}  {doc}")

질문: 법인카드 영수증은 언제까지 제출해야 하나요?

=== 유사도 순위 ===
  0.653  법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.
  0.228  개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.
  0.227  재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.
  0.187  신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.
  0.098  장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.


> 위 결과에서 상위 1~2 개 문서를 LLM 에 "참고 자료" 로 같이 넣으면 그게 가장 단순한 RAG.

## 5. 임베딩 모델 비교 표 (2026-06)

| 모델 | 기본 차원 | 한국어 품질 | 가격(1M input tokens, standard) |
|---|---|---|---|
| `text-embedding-3-small` | 1536 (조절 가능) | 좋음 | 0.02 달러 |
| `text-embedding-3-large` | 3072 (조절 가능) | 우수 | 0.13 달러 |
| `bge-m3` (오픈) | 1024 | 매우 좋음 (한국어 최적화) | 무료 (셀프 호스팅) |
| `KoSimCSE` (오픈) | 768 | 한국어만 | 무료 |


> `dimensions` 옵션은 `text-embedding-3` 계열에서 지원. 차원을 줄이면 저장·검색 메모리는 줄지만 품질이 살짝 떨어질 수 있음.

비용·품질 trade-off. 한국어 강의 자료는 `text-embedding-3-small` 로 충분.


## 6. 정리

- RAG = Retrieve (외부 검색) + Augment (프롬프트 보강) + Generate (LLM 응답)
- 텍스트 → 벡터 → 코사인 유사도 비교
- 다음 노트북부터 5단계를 하나씩 구현

## [실습]
1. `embeddings.embed_query("카드값 증빙은 언제 올려야 해?")` 와 위 docs 의 유사도 확인.
2. text-embedding-3-large 로 바꿔 같은 비교 / 점수 차이?
3. `dimensions=512` 옵션으로 임베딩 차원 줄이고 결과 비교 (저장 비용 절감 효과).



In [19]:
# 1. `embeddings.embed_query("카드값 증빙은 언제 올려야 해?")` 와 위 docs 의 유사도 확인.

import pandas as pd

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

query = "카드값 증빙은 언제 올려야 해?"

q_vector = embeddings.embed_query(query)
doc_vectors = embeddings.embed_documents(docs)

rows = []

for i, (doc, doc_vector) in enumerate(zip(docs, doc_vectors), start=1):
    rows.append({
        "rank_target": i,
        "score": cosine_sim(q_vector, doc_vector),
        "doc": doc,
    })

result_df = pd.DataFrame(rows).sort_values("score", ascending=False)
result_df

,rank_target,score,doc
1,2,0.479247,법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에...
3,4,0.234254,"장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비..."
4,5,0.231094,"재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받..."
2,3,0.159620,"개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연..."
0,1,0.148637,"신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사..."


In [20]:
# 2. text-embedding-3-large 로 바꿔 같은 비교 / 점수 차이?

def search_docs(model, dimensions=None):
    kwargs = {"model": model}
    if dimensions is not None:
        kwargs["dimensions"] = dimensions

    emb = OpenAIEmbeddings(**kwargs)

    q_vec = emb.embed_query(query)
    doc_vecs = emb.embed_documents(docs)

    rows = []
    for i, (doc, doc_vec) in enumerate(zip(docs, doc_vecs), start=1):
        rows.append({
            "model": model,
            "dimensions": dimensions or "default",
            "actual_dim": len(q_vec),
            "doc_no": i,
            "score": cosine_sim(q_vec, doc_vec),
            "doc": doc,
        })

    return pd.DataFrame(rows).sort_values("score", ascending=False)

small_df = search_docs("text-embedding-3-small")
large_df = search_docs("text-embedding-3-large")

compare_df = pd.concat([small_df, large_df], ignore_index=True)
compare_df

,model,dimensions,actual_dim,doc_no,score,doc
0,text-embedding-3-small,default,1536,2,0.479212,법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에...
1,text-embedding-3-small,default,1536,4,0.234455,"장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비..."
2,text-embedding-3-small,default,1536,5,0.231088,"재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받..."
3,text-embedding-3-small,default,1536,3,0.159518,"개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연..."
4,text-embedding-3-small,default,1536,1,0.148611,"신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사..."
5,text-embedding-3-large,default,3072,2,0.486997,법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에...
6,text-embedding-3-large,default,3072,4,0.265758,"장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비..."
7,text-embedding-3-large,default,3072,3,0.221312,"개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연..."
8,text-embedding-3-large,default,3072,5,0.216612,"재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받..."
9,text-embedding-3-large,default,3072,1,0.214054,"신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사..."


In [21]:
# 3. `dimensions=512` 옵션으로 임베딩 차원 줄이고 결과 비교 (저장 비용 절감 효과).
small_512_df = search_docs("text-embedding-3-small", dimensions=512)
large_512_df = search_docs("text-embedding-3-large", dimensions=512)

dimension_compare_df = pd.concat(
    [small_df, small_512_df, large_df, large_512_df],
    ignore_index=True,
)

dimension_compare_df

,model,dimensions,actual_dim,doc_no,score,doc
0,text-embedding-3-small,default,1536,2,0.479212,법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에...
1,text-embedding-3-small,default,1536,4,0.234455,"장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비..."
2,text-embedding-3-small,default,1536,5,0.231088,"재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받..."
3,text-embedding-3-small,default,1536,3,0.159518,"개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연..."
4,text-embedding-3-small,default,1536,1,0.148611,"신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사..."
5,text-embedding-3-small,512,512,2,0.507762,법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에...
6,text-embedding-3-small,512,512,5,0.268003,"재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받..."
7,text-embedding-3-small,512,512,4,0.233687,"장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비..."
8,text-embedding-3-small,512,512,3,0.202476,"개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연..."
9,text-embedding-3-small,512,512,1,0.196379,"신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사..."


In [22]:
# 점수 비교표
pivot_df = dimension_compare_df.pivot_table(
    index="doc_no",
    columns=["model", "dimensions"],
    values="score",
)

pivot_df["doc"] = docs
pivot_df

model      text-embedding-3-large           text-embedding-3-small            \
dimensions                    512   default                    512   default   
doc_no                                                                         
1                        0.242538  0.214054               0.196379  0.148611   
2                        0.525769  0.486997               0.507762  0.479212   
3                        0.238540  0.221312               0.202476  0.159518   
4                        0.251483  0.265758               0.233687  0.234455   
5                        0.262000  0.216612               0.268003  0.231088   

model                                                     doc  
dimensions                                                     
doc_no                                                         
1           신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사...  
2           법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에...  
3           개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연...  
4           장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비...  
5           재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받...

In [23]:
# 저장 공간 차이
storage_df = (
    dimension_compare_df[["model", "dimensions", "actual_dim"]]
    .drop_duplicates()
    .copy()
)

storage_df["bytes_per_vector_float32"] = storage_df["actual_dim"] * 4
storage_df["kb_per_1000_vectors"] = storage_df["bytes_per_vector_float32"] * 1000 / 1024
storage_df["mb_per_1m_vectors"] = storage_df["bytes_per_vector_float32"] * 1_000_000 / 1024 / 1024

storage_df

,model,dimensions,actual_dim,bytes_per_vector_float32,kb_per_1000_vectors,mb_per_1m_vectors
0,text-embedding-3-small,default,1536,6144,6000.0,5859.375
5,text-embedding-3-small,512,512,2048,2000.0,1953.125
10,text-embedding-3-large,default,3072,12288,12000.0,11718.750
15,text-embedding-3-large,512,512,2048,2000.0,1953.125
